In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
pd.set_option('display.max_columns', None)

In [ ]:
!pip install pandas pyarrow pycountry -q
# import gbd_burden

In [ ]:
#!/usr/bin/env python3
"""
gbd_burden.py  —  Google Colab
==============================
Tier-3 BURDEN layer (human-disease side) for Paper 1. Turns the IHME GBD Results
Tool CSV export into a country-year burden panel and joins it to your corpus by
(study country, publication year) — the burden of the place a paper studies, in
the year it studied it.

INPUT: the GBD CSV you exported (see the export instructions). Expected columns:
  location_name, year, cause_name, measure_name, metric_name, val, upper, lower
  (some exports also carry location_id, cause_id, sex_name, age_name — ignored)



Go to the GBD Results Tool (vizhub.healthdata.org/gbd-results). Select exactly these so the download is small and the join is clean:

Measure: DALYs (Disability-Adjusted Life Years). Also tick Deaths and Incidence if you want them as robustness outcomes — cheap to grab now.
Metric: both Rate (per 100,000 — this is what you compare across countries of different sizes) and Number (absolute burden, useful for the "where is the burden concentrated" map). Rate is the primary.
Cause: the tick-borne human diseases GBD actually codes. This is the catch — GBD does not have a clean "tick-borne diseases" bucket, so you select the specific causes it does track and treat the set as your TBD burden proxy. The relevant ones: Encephalitis (captures TBE), and the broader vector/zoonotic parents where TBDs sit — but Lyme, CCHF, and spotted fevers are largely folded into aggregate categories, not itemized. Grab Encephalitis, and the "Neglected tropical diseases and malaria" parent plus "Other neglected tropical diseases" as the closest itemized homes. Note this limitation now — it's a real one you'll disclose.
Location: all countries (not regions/aggregates — you'll join on country).
Age: All ages. Sex: Both.
Year: the full range available (1990 onward) so it aligns to your corpus years.



WHAT IT DOES
  1. filters to DALYs (keeps Deaths/Incidence too if present, as extra outcomes)
  2. maps GBD location_name -> ISO2 (via pycountry, with a manual patch table for
     GBD's naming quirks)
  3. pivots to one row per (iso2, year) with a column per cause x measure x metric
  4. builds a combined 'tbd_daly_rate' = sum of the selected TBD-proxy causes
  5. joins onto your corpus by primary study country + oa_year (nearest-prior-year
     fallback, since GBD is annual but your papers span all years)

CAVEAT baked into the output: GBD has no clean 'tick-borne diseases' cause. You
selected the closest itemized causes (encephalitis, NTD parents). The column
'tbd_daly_rate' is a PROXY; report which causes compose it (printed at the end).

SETUP
  !pip install pandas pyarrow pycountry -q


gbd_burden.py  —  Google Colab   (multi-zip IHME export edition)
================================================================
Tier-3 BURDEN layer for Paper 1. IHME splits a large query into MANY numbered
zips (each holding one CSV). This reads EVERY zip in a folder, concatenates,
and joins to your corpus by (study country, publication year).

Matches your actual export columns:
  population_group_name, measure_name (Deaths / DALYs / Incidence ...),
  location_name, cause_name, metric_name (Number / Rate), year, val, upper, lower

SETUP
  !pip install pandas pyarrow pycountry -q
"""

import glob, io, sys, zipfile
import pandas as pd
import pycountry
import sys
from pathlib import Path

# =========================================================================== #
# CONFIG                                                                        #
# =========================================================================== #
GBD_ZIP_DIR = "/content/drive/MyDrive/COGNOSX/ticks/external_data/GBD"   # folder of IHME-GBD_*.zip
IN_PARQUET  = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/with_study_location.parquet"
OUT_PANEL   = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/external_data/gbd_burden_country_year.parquet"
OUT_MERGED  = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/with_burden.parquet"

Path(OUT_PANEL).parent.mkdir(parents=True, exist_ok=True)


# EDIT after the first run prints what's actually present:
MEASURE_WANTED = "DALYs (Disability-Adjusted Life Years)"  # or "Deaths" if DALYs absent
METRIC_WANTED  = "Rate"                                    # Rate = per-100k, comparable
TBD_PROXY_CAUSES = ["Encephalitis", "Other neglected tropical diseases"]


def S(x):
    return "" if x is None or (isinstance(x, float) and pd.isna(x)) or x is pd.NA else str(x)


# =========================================================================== #
# GBD location_name -> ISO2                                                    #
# =========================================================================== #
GBD_NAME_PATCH = {
    "Bolivia (Plurinational State of)": "BO", "Bolivia": "BO",
    "Democratic Republic of the Congo": "CD", "Congo": "CG",
    "Côte d'Ivoire": "CI", "Cote d'Ivoire": "CI",
    "Iran (Islamic Republic of)": "IR", "Iran": "IR",
    "Republic of Korea": "KR", "Democratic People's Republic of Korea": "KP",
    "Lao People's Democratic Republic": "LA",
    "Republic of Moldova": "MD", "Russian Federation": "RU",
    "Syrian Arab Republic": "SY", "United Republic of Tanzania": "TZ",
    "Tanzania (United Republic of)": "TZ",
    "United States of America": "US", "Venezuela (Bolivarian Republic of)": "VE",
    "Viet Nam": "VN", "Türkiye": "TR", "Turkey": "TR",
    "United Kingdom": "GB", "Cabo Verde": "CV", "The Gambia": "GM",
    "Eswatini": "SZ", "Taiwan (Province of China)": "TW",
    "Micronesia (Federated States of)": "FM", "Czechia": "CZ",
    "Palestine": "PS", "United States Virgin Islands": "VI",
}

def name_to_iso2(name):
    n = S(name).strip()
    if n in GBD_NAME_PATCH:
        return GBD_NAME_PATCH[n]
    try:
        return pycountry.countries.lookup(n).alpha_2
    except LookupError:
        return ""


def iso2_to_iso3(x):
    try: return pycountry.countries.lookup(x).alpha_3
    except LookupError: return ""


# =========================================================================== #
# 1. read EVERY zip in the folder, concatenate all inner CSVs                  #
# =========================================================================== #
def load_all_zips():
    zips = sorted(glob.glob(f"{GBD_ZIP_DIR}/*.zip"))
    if not zips:
        raise SystemExit(f"[gbd] no .zip files in {GBD_ZIP_DIR}")
    print(f"[gbd] {len(zips)} zip files", file=sys.stderr)
    frames = []
    for z in zips:
        with zipfile.ZipFile(z) as zf:
            for member in zf.namelist():
                if member.lower().endswith(".csv"):
                    with zf.open(member) as fh:
                        frames.append(pd.read_csv(io.BytesIO(fh.read()), dtype=str))
    g = pd.concat(frames, ignore_index=True)
    g.columns = [c.strip().lower() for c in g.columns]
    g = g.drop_duplicates()
    for c in ("val", "upper", "lower"):
        if c in g.columns:
            g[c] = pd.to_numeric(g[c], errors="coerce")
    g["year"] = pd.to_numeric(g["year"], errors="coerce").astype("Int64")
    print(f"[gbd] {len(g):,} rows concatenated", file=sys.stderr)
    print(f"[gbd] measures present : {sorted(g['measure_name'].dropna().unique())}",
          file=sys.stderr)
    print(f"[gbd] metrics present  : {sorted(g['metric_name'].dropna().unique())}",
          file=sys.stderr)
    print(f"[gbd] causes present   : {sorted(g['cause_name'].dropna().unique())}",
          file=sys.stderr)
    print(f"[gbd] year range       : {int(g['year'].min())}-{int(g['year'].max())}",
          file=sys.stderr)
    return g


# =========================================================================== #
# 2. filter to wanted measure+metric, pivot to country-year TBD proxy         #
# =========================================================================== #
def build_panel(g):
    sub = g[(g["measure_name"] == MEASURE_WANTED) & (g["metric_name"] == METRIC_WANTED)].copy()
    if sub.empty:
        print(f"[gbd] WARNING: no rows for measure='{MEASURE_WANTED}' "
              f"metric='{METRIC_WANTED}'. Edit MEASURE_WANTED/METRIC_WANTED to one "
              f"of the values printed above.", file=sys.stderr)
    sub["iso2"] = sub["location_name"].map(name_to_iso2)
    unmatched = sorted(set(sub.loc[sub["iso2"] == "", "location_name"]))
    if unmatched:
        print(f"[gbd] {len(unmatched)} location_names unmapped (regions/aggregates): "
              f"{unmatched[:10]}{' ...' if len(unmatched)>10 else ''}", file=sys.stderr)
    sub = sub[sub["iso2"] != ""]

    panel = sub.pivot_table(index=["iso2", "year"], columns="cause_name",
                            values="val", aggfunc="first").reset_index()
    panel["iso3"] = panel["iso2"].map(iso2_to_iso3) #
    proxy = [c for c in TBD_PROXY_CAUSES if c in panel.columns]
    panel["tbd_daly_rate"] = panel[proxy].sum(axis=1, min_count=1) if proxy else pd.NA
    print(f"[gbd] TBD proxy from: {proxy}", file=sys.stderr)
    print(f"[gbd] panel: {panel['iso2'].nunique()} countries, {len(panel):,} rows",
          file=sys.stderr)
    return panel


# =========================================================================== #
# 3. join onto corpus by (primary study country, publication year)            #
# =========================================================================== #
def attach(df, panel):
    idx = panel.set_index(["iso2", "year"])
    have = set(panel["iso2"])

    def row(r):
        sc = [c for c in S(r.get("study_countries")).split(";") if c]
        year = pd.to_numeric(r.get("oa_year"), errors="coerce")
        if not sc or pd.isna(year) or sc[0] not in have:
            return pd.Series({"tbd_daly_rate": None, "burden_iso2": "", "burden_year_used": None})
        iso2, yr = sc[0], int(year)
        val, used = None, None
        if (iso2, yr) in idx.index:
            val = idx.loc[(iso2, yr), "tbd_daly_rate"]; used = yr
        if pd.isna(val) and iso2 in idx.index.get_level_values(0):
            prior = idx.loc[iso2, "tbd_daly_rate"].loc[:yr].dropna()
            if len(prior):
                val, used = prior.iloc[-1], int(prior.index[-1])
        return pd.Series({"tbd_daly_rate": val, "burden_iso2": iso2, "burden_year_used": used})

    out = pd.concat([df, df.apply(row, axis=1)], axis=1)
    out.to_parquet(OUT_MERGED, index=False)
    print(f"[write] {OUT_MERGED}", file=sys.stderr)
    print(f"[coverage] papers with a TBD burden value: "
          f"{out['tbd_daly_rate'].notna().mean():.1%}", file=sys.stderr)
    return out


# =========================================================================== #
if __name__ == "__main__":
    g = load_all_zips()
    panel = build_panel(g)
    panel.to_parquet(OUT_PANEL, index=False)
    print(f"[write] {OUT_PANEL}", file=sys.stderr)

    df_all = pd.read_parquet(IN_PARQUET)
    df = df_all[df_all["llm_label"].notna()]
    print(f"[in] {len(df):,} rows", file=sys.stderr)
    merge = attach(df, panel)


In [ ]:
panel.describe()

In [ ]:
panel

In [ ]:
merge

In [ ]:
# #!/usr/bin/env python3
# """
# gbd_burden.py  —  Google Colab
# ==============================
# Tier-3 BURDEN layer (human-disease side) for Paper 1. Turns the IHME GBD Results
# Tool CSV export into a country-year burden panel and joins it to your corpus by
# (study country, publication year) — the burden of the place a paper studies, in
# the year it studied it.

# INPUT: the GBD CSV you exported (see the export instructions). Expected columns:
#   location_name, year, cause_name, measure_name, metric_name, val, upper, lower
#   (some exports also carry location_id, cause_id, sex_name, age_name — ignored)



# Go to the GBD Results Tool (vizhub.healthdata.org/gbd-results). Select exactly these so the download is small and the join is clean:

# Measure: DALYs (Disability-Adjusted Life Years). Also tick Deaths and Incidence if you want them as robustness outcomes — cheap to grab now.
# Metric: both Rate (per 100,000 — this is what you compare across countries of different sizes) and Number (absolute burden, useful for the "where is the burden concentrated" map). Rate is the primary.
# Cause: the tick-borne human diseases GBD actually codes. This is the catch — GBD does not have a clean "tick-borne diseases" bucket, so you select the specific causes it does track and treat the set as your TBD burden proxy. The relevant ones: Encephalitis (captures TBE), and the broader vector/zoonotic parents where TBDs sit — but Lyme, CCHF, and spotted fevers are largely folded into aggregate categories, not itemized. Grab Encephalitis, and the "Neglected tropical diseases and malaria" parent plus "Other neglected tropical diseases" as the closest itemized homes. Note this limitation now — it's a real one you'll disclose.
# Location: all countries (not regions/aggregates — you'll join on country).
# Age: All ages. Sex: Both.
# Year: the full range available (1990 onward) so it aligns to your corpus years.



# WHAT IT DOES
#   1. filters to DALYs (keeps Deaths/Incidence too if present, as extra outcomes)
#   2. maps GBD location_name -> ISO2 (via pycountry, with a manual patch table for
#      GBD's naming quirks)
#   3. pivots to one row per (iso2, year) with a column per cause x measure x metric
#   4. builds a combined 'tbd_daly_rate' = sum of the selected TBD-proxy causes
#   5. joins onto your corpus by primary study country + oa_year (nearest-prior-year
#      fallback, since GBD is annual but your papers span all years)

# CAVEAT baked into the output: GBD has no clean 'tick-borne diseases' cause. You
# selected the closest itemized causes (encephalitis, NTD parents). The column
# 'tbd_daly_rate' is a PROXY; report which causes compose it (printed at the end).

# SETUP
#   !pip install pandas pyarrow pycountry -q
# """

# import sys
# import pandas as pd
# import pycountry

# # =========================================================================== #
# # CONFIG                                                                        #
# # =========================================================================== #
# GBD_CSV     = "/content/drive/MyDrive/.../gbd_daly_export.csv"
# IN_PARQUET  = "/content/drive/MyDrive/.../with_capacity.parquet"
# OUT_PANEL   = "/content/drive/MyDrive/.../gbd_burden_country_year.parquet"
# OUT_MERGED  = "/content/drive/MyDrive/.../with_burden.parquet"

# # the causes you selected as the TBD proxy — EDIT to match your export exactly
# # (print at the end shows the cause_names actually present so you can reconcile)
# TBD_PROXY_CAUSES = [
#     "Encephalitis",
#     "Other neglected tropical diseases",
#     # add/rename to whatever your export contains, e.g. "Meningitis" as robustness
# ]
# PRIMARY_MEASURE = "DALYs (Disability-Adjusted Life Years)"
# PRIMARY_METRIC  = "Rate"


# def S(x):
#     return "" if x is None or (isinstance(x, float) and pd.isna(x)) or x is pd.NA else str(x)


# # =========================================================================== #
# # GBD location_name -> ISO2                                                    #
# # =========================================================================== #
# # pycountry handles most; GBD uses some non-standard names — patch here.
# GBD_NAME_PATCH = {
#     "Bolivia (Plurinational State of)": "BO", "Bolivia": "BO",
#     "Democratic Republic of the Congo": "CD", "Congo": "CG",
#     "Côte d'Ivoire": "CI", "Cote d'Ivoire": "CI",
#     "Iran (Islamic Republic of)": "IR", "Iran": "IR",
#     "Republic of Korea": "KR", "South Korea": "KR",
#     "Democratic People's Republic of Korea": "KP", "North Korea": "KP",
#     "Lao People's Democratic Republic": "LA", "Laos": "LA",
#     "Republic of Moldova": "MD", "Moldova": "MD",
#     "Russian Federation": "RU", "Russia": "RU",
#     "Syrian Arab Republic": "SY", "Syria": "SY",
#     "United Republic of Tanzania": "TZ", "Tanzania": "TZ",
#     "United States of America": "US", "United States": "US",
#     "Venezuela (Bolivarian Republic of)": "VE", "Venezuela": "VE",
#     "Viet Nam": "VN", "Vietnam": "VN",
#     "Türkiye": "TR", "Turkey": "TR",
#     "United Kingdom": "GB", "United Kingdom of Great Britain and Northern Ireland": "GB",
#     "Cabo Verde": "CV", "Cape Verde": "CV",
#     "The Gambia": "GM", "Gambia": "GM",
#     "Eswatini": "SZ", "Swaziland": "SZ",
#     "Taiwan (Province of China)": "TW",
#     "Micronesia (Federated States of)": "FM",
#     "Tanzania (United Republic of)": "TZ",
# }

# def name_to_iso2(name):
#     n = S(name).strip()
#     if n in GBD_NAME_PATCH:
#         return GBD_NAME_PATCH[n]
#     try:
#         return pycountry.countries.lookup(n).alpha_2
#     except LookupError:
#         return ""


# # =========================================================================== #
# # 1. load + pivot the GBD export into a country-year panel                    #
# # =========================================================================== #
# def build_panel():
#     g = pd.read_csv(GBD_CSV, dtype=str)
#     g.columns = [c.strip().lower() for c in g.columns]
#     for c in ("val", "upper", "lower"):
#         if c in g.columns:
#             g[c] = pd.to_numeric(g[c], errors="coerce")
#     g["year"] = pd.to_numeric(g["year"], errors="coerce").astype("Int64")
#     g["iso2"] = g["location_name"].map(name_to_iso2)
#     unmatched = sorted(set(g.loc[g["iso2"] == "", "location_name"]))
#     if unmatched:
#         print(f"[gbd] {len(unmatched)} location_names didn't map (likely regions/"
#               f"aggregates): {unmatched[:12]}{' ...' if len(unmatched)>12 else ''}",
#               file=sys.stderr)
#     g = g[g["iso2"] != ""]

#     # one column per cause x measure x metric
#     g["colkey"] = (g["cause_name"].str.strip() + " | " +
#                    g["measure_name"].str.strip() + " | " +
#                    g["metric_name"].str.strip())
#     panel = g.pivot_table(index=["iso2", "year"], columns="colkey",
#                           values="val", aggfunc="first").reset_index()

#     # combined TBD proxy = sum of selected causes at PRIMARY measure+metric
#     proxy_cols = [f"{c} | {PRIMARY_MEASURE} | {PRIMARY_METRIC}"
#                   for c in TBD_PROXY_CAUSES
#                   if f"{c} | {PRIMARY_MEASURE} | {PRIMARY_METRIC}" in panel.columns]
#     if proxy_cols:
#         panel["tbd_daly_rate"] = panel[proxy_cols].sum(axis=1, min_count=1)
#     else:
#         panel["tbd_daly_rate"] = pd.NA
#         print("[gbd] WARNING: none of TBD_PROXY_CAUSES matched — check cause names",
#               file=sys.stderr)

#     present = sorted(set(g["cause_name"]))
#     print(f"[gbd] causes present in export: {present}", file=sys.stderr)
#     print(f"[gbd] TBD proxy built from: {[c.split(' | ')[0] for c in proxy_cols]}",
#           file=sys.stderr)
#     print(f"[gbd] panel: {panel['iso2'].nunique()} countries x years, "
#           f"{len(panel):,} rows", file=sys.stderr)
#     return panel, proxy_cols


# # =========================================================================== #
# # 2. join onto the corpus by (primary study country, publication year)        #
# # =========================================================================== #
# def attach(df, panel):
#     idx = panel.set_index(["iso2", "year"])
#     have_iso = set(panel["iso2"])

#     def row(r):
#         sc = [c for c in S(r.get("study_countries")).split(";") if c]
#         year = pd.to_numeric(r.get("oa_year"), errors="coerce")
#         if not sc or pd.isna(year) or sc[0] not in have_iso:
#             return pd.Series({"tbd_daly_rate": None, "burden_iso2": "", "burden_year_used": None})
#         iso2, yr = sc[0], int(year)
#         val, used = None, None
#         if (iso2, yr) in idx.index:
#             val = idx.loc[(iso2, yr), "tbd_daly_rate"]; used = yr
#         if pd.isna(val):
#             sub = idx.loc[iso2, "tbd_daly_rate"] if iso2 in idx.index.get_level_values(0) else None
#             if sub is not None:
#                 prior = sub.loc[:yr].dropna()
#                 if len(prior):
#                     val, used = prior.iloc[-1], int(prior.index[-1])
#         return pd.Series({"tbd_daly_rate": val, "burden_iso2": iso2, "burden_year_used": used})

#     out = pd.concat([df, df.apply(row, axis=1)], axis=1)
#     out.to_parquet(OUT_MERGED, index=False)
#     print(f"[write] {OUT_MERGED}", file=sys.stderr)
#     cov = out["tbd_daly_rate"].notna().mean()
#     print(f"[coverage] papers with a TBD burden value: {cov:.1%}", file=sys.stderr)
#     return out


# # =========================================================================== #
# if __name__ == "__main__":
#     panel, proxy_cols = build_panel()
#     panel.to_parquet(OUT_PANEL, index=False)
#     print(f"[write] {OUT_PANEL}", file=sys.stderr)

#     df = pd.read_parquet(IN_PARQUET)
#     print(f"[in] {len(df):,} rows", file=sys.stderr)
#     attach(df, panel)